# CatBoost — SBA Loan Default Prediction

CatBoost is a gradient boosting library developed by Yandex that builds an ensemble of decision trees sequentially, where each new tree corrects the errors of the previous ones. It is particularly known for:

- **Native categorical feature support** — no manual one-hot encoding required
- **Ordered boosting** — reduces overfitting by using a permutation-based technique
- **Fast training and prediction** with GPU support
- **class_weights** parameter to directly address class imbalance

For this SBA loan default prediction task, CatBoost's ordered boosting and flexible class weighting make it well-suited for handling the ~17% default rate imbalance.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve, precision_recall_curve)
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

# ── Load preprocessed data ───────────────────────────────────────────────────
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Default rate — Train: {y_train.mean():.2%}, Test: {y_test.mean():.2%}")

## 1. Model Training

CatBoost builds 500 gradient-boosted trees sequentially, each correcting residual errors from the previous iteration. The key hyperparameters used here:

- `iterations=500` — number of boosting rounds
- `learning_rate=0.1` — step size shrinkage to prevent overfitting
- `depth=6` — maximum tree depth (controls model complexity)
- `eval_metric="AUC"` — optimises for discriminative power, important for imbalanced data
- `class_weights=[1, 5]` — gives the default class (1) five times more weight than non-default (0), compensating for the ~17% imbalance
- `verbose=0` — suppresses per-iteration output

In [ ]:
%%time
# ── Install CatBoost if not already installed ─────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "catboost", "-q"])

# ── Train CatBoost ────────────────────────────────────────────────────────────
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=500,          # number of boosting rounds
    learning_rate=0.1,       # step size — smaller = slower but better generalisation
    depth=6,                 # max tree depth
    eval_metric="AUC",       # track AUC during training
    class_weights=[1, 5],    # weight class 1 (default) 5x more than class 0
    random_seed=42,
    verbose=0                # suppress iteration-level output
)

model.fit(X_train, y_train)
print("Model training complete.")

## 2. Predictions

Generating hard class labels and default probability scores from the trained CatBoost model.

In [ ]:
# ── Generate predictions ──────────────────────────────────────────────────────
y_pred = model.predict(X_test)              # hard class labels (0 or 1)
y_prob = model.predict_proba(X_test)[:, 1]  # probability of default (class 1)

print(f"Predicted defaults: {y_pred.sum()} / {len(y_pred)} ({y_pred.mean():.2%})")
print(f"Actual defaults:    {y_test.sum()} / {len(y_test)} ({y_test.mean():.2%})")

## 3. Metric Analysis

### Which metric matters most for this problem?

This is an **imbalanced classification problem** (~17% default rate), which makes metric choice critical:

| Metric | Why it matters here |
|--------|-------------------|
| **Recall** | Most important. Measures how many actual defaults we caught. Missing a default (false negative) means approving a loan that will not be repaid — a direct financial loss. |
| **ROC-AUC** | Measures the model's overall discrimination ability across all thresholds. Robust to class imbalance and useful for comparing models. |
| **PR-AUC** | Even more informative than ROC-AUC on imbalanced data — focuses specifically on the minority (default) class performance. |
| **F1 Score** | Harmonic mean of Precision and Recall. Useful when you need a single number that balances both, but does not capture the full picture. |
| **Accuracy** | **Misleading here.** A naive model that predicts "no default" for every loan would achieve ~83% accuracy — yet it would be completely useless for fraud/default detection. |

**Business context:** In SBA loan default prediction, a **false negative** (predicting "paid in full" when the loan actually defaults) is far more costly than a **false positive** (flagging a good loan as risky). Therefore, we prioritise **Recall** and **ROC-AUC** when evaluating model quality.

> CatBoost was trained with `eval_metric="AUC"` and `class_weights=[1, 5]`, directly optimising for discriminative power while compensating for class imbalance.

In [ ]:
# ── Compute and display all metrics ──────────────────────────────────────────
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Paid in Full", "Defaulted"]))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Paid in Full", "Defaulted"],
            yticklabels=["Paid in Full", "Defaulted"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — CatBoost")
plt.tight_layout()
plt.show()

# The bottom-left cell (false negatives) represents defaults the model missed.
# Minimising this cell is the primary business objective.

In [ ]:
# ── ROC Curve ────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"ROC Curve (AUC = {auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random Classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — CatBoost")
ax.legend()
plt.tight_layout()
plt.show()

# A higher AUC means the model better separates defaults from non-defaults
# across all possible decision thresholds.

In [ ]:
# ── Precision-Recall Curve ───────────────────────────────────────────────────
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall_vals, precision_vals, color="salmon", lw=2, label="CatBoost")
ax.axhline(y=y_test.mean(), color="gray", linestyle="--",
           label=f"Baseline (random) = {y_test.mean():.2%}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve — CatBoost")
ax.legend()
plt.tight_layout()
plt.show()

# The dashed baseline represents a random classifier at the observed default rate.
# A good model keeps precision well above baseline at high recall values.

## 4. Feature Importance

CatBoost computes feature importance using the **PredictionValuesChange** method by default — it measures how much each feature affects the model's output values on average across all trees. Higher scores indicate features that most strongly influence predictions.

In [ ]:
# ── Feature Importance bar chart ─────────────────────────────────────────────
# get_feature_importance() returns PredictionValuesChange scores by default
importances = pd.Series(model.get_feature_importance(), index=X_train.columns)
top_20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
top_20.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Feature Importance (PredictionValuesChange)")
ax.set_title("Top 20 Feature Importances — CatBoost")
plt.tight_layout()
plt.show()

# Features at the top most strongly drive the model's predictions.
# Unlike Gini-based importance, CatBoost's metric is less biased towards
# high-cardinality features.